In [1]:
import numpy as np
from scipy.integrate import nquad
from numpy.polynomial.legendre import leggauss

\begin{equation}
    f1 = \int_{0}^{1}\int_{-\infty}^{\infty}  e^{-y^2} x^2 dydx= \frac{\sqrt{\pi}}{3} 
\end{equation}

\begin{equation}
    f2 = \int_{0}^{\pi}\int_{-\infty}^{\infty}  e^{-|y|} \sin(x) dy dx= 4
\end{equation}

In [2]:
#testing functions 

def f1(x, y):
    return np.exp(-y**2) * x**2

def f2(x, y):
    return np.exp(-np.abs(y)) * np.sin(x)    

In [3]:
# Inner integral: finite interval with Gauss-Legendre quadrature
def integrate_x(y, a, b, n=20):
    # Gauss-Legendre nodes and weights on [-1, 1]
    nodes, weights = leggauss(n)
    # Map nodes from [-1, 1] to [a, b]
    mapped_nodes = 0.5 * (nodes * (b - a) + (b + a))
    mapped_weights = 0.5 * (b - a) * weights
    # Evaluate f(x, y) at the nodes
    values = f2(mapped_nodes, y)
    return np.sum(values * mapped_weights)

# Outer integral: infinite range with nquad
def hybrid_integral(a, b):
    def integrand(y):
        return integrate_x(y, a, b, n=20)
    result, err = nquad(lambda y: integrand(y), [[-np.inf, np.inf]])
    return result, err

# Example of f1 
res, err = hybrid_integral(0, np.pi)
print("Integral result:", res, " ± ", err)


Integral result: 3.999999999999997  ±  2.3370427715721285e-10


\begin{equation}
    f3 = \int_{0}^{1}\int_{0}^{1}\int_{-\infty}^{\infty}\int_{-\infty}^{\infty}  (x+y)e^{-(v_x^2 + v_y^2)} dv_x dv_y dx dy= \pi
\end{equation}

\begin{equation}
    f4 = \int_{0}^{\pi}\int_{0}^{\pi}\int_{-\infty}^{\infty}\int_{-\infty}^{\infty}  \sin(x)\cos(y)\frac{1}{(1 + v_x^2)(1 + v_y^2)} dv_x dv_y dx dy= 0
\end{equation}

In [4]:
def f3(x,y,vx,vy):
    return (x + y) * np.exp(-(vx**2 + vy**2))


def f4(x,y,vx,vy):
    return np.sin(x) * np.cos(y) * 1/((1 + vx**2)*(1 + vy**2))

In [5]:
# Finite-range Gauss–Legendre integration over x,y
def integrate_xy(vx, vy, ax, bx, ay, by, n=20):
    # Gauss–Legendre nodes and weights
    nodes, weights = leggauss(n)

    # Map nodes from [-1, 1] to [ax, bx] and [ay, by]
    x_nodes = 0.5 * (nodes * (bx - ax) + (bx + ax))
    y_nodes = 0.5 * (nodes * (by - ay) + (by + ay))
    x_weights = 0.5 * (bx - ax) * weights
    y_weights = 0.5 * (by - ay) * weights

    # Tensor product quadrature
    total = 0.0
    for i, xi in enumerate(x_nodes):
        for j, yj in enumerate(y_nodes):
            total += f4(xi, yj, vx, vy) * x_weights[i] * y_weights[j]
    return total

# Outer integral over infinite vx, vy using nquad
def hybrid_integral(ax, bx, ay, by, n=20):
    def integrand(vx, vy):
        return integrate_xy(vx, vy, ax, bx, ay, by, n=n)

    # nquad over (-∞, ∞) × (-∞, ∞)
    result, err = nquad(integrand, [[-np.inf, np.inf], [-np.inf, np.inf]])
    return result, err

# Example usage
res, err = hybrid_integral(0, np.pi, 0, np.pi, n=20)
print("Integral result:", res, " ± ", err)

Integral result: 3.612308891222126e-15  ±  6.566233513080693e-16


In [6]:
def f5(x,y,z,vx,vy,vz):
    return x * y**2 * z**2 * np.exp(-(vx**2 + vy**2 + vz**2))

In [7]:
# Finite-range Gauss–Legendre integration over x,y
def integrate_xy(vx, vy, vz, ax, bx, ay, by, az, bz, n=20):
    # Gauss–Legendre nodes and weights
    nodes, weights = leggauss(n)

    # Map nodes from [-1, 1] to [ax, bx] and [ay, by]
    x_nodes = 0.5 * (nodes * (bx - ax) + (bx + ax))
    y_nodes = 0.5 * (nodes * (by - ay) + (by + ay))
    z_nodes = 0.5 * (nodes * (bz - az) + (bz + az))
    x_weights = 0.5 * (bx - ax) * weights
    y_weights = 0.5 * (by - ay) * weights
    z_weights = 0.5 * (bz - az) * weights

    # Tensor product quadrature
    total = 0.0
    for i, xi in enumerate(x_nodes):
        for j, yj in enumerate(y_nodes):
            for k, zk in enumerate(z_nodes):
                total += f5(xi, yj, zk, vx, vy, vz) * x_weights[i] * y_weights[j] * z_weights[k]
    return total

# Outer integral over infinite vx, vy using nquad
def hybrid_integral(ax, bx, cx, ay, by, cy, n=20):
    def integrand(vx, vy, vz):
        return integrate_xy(vx, vy, vz, ax, bx, cx, ay, by, cy, n=n)

    # nquad over (-∞, ∞) × (-∞, ∞)
    result, err = nquad(integrand, [[-np.inf, np.inf], [-np.inf, np.inf], [-np.inf, np.inf]])
    return result, err

# Example usage
#res, err = hybrid_integral(0, 1, 0, np.pi, -1, 1, n=20)
print("Integral result:", res, " ± ", err)

Integral result: 3.612308891222126e-15  ±  6.566233513080693e-16


## Spheric Integration

### Testing spheric integration in velocities and cube in positions

In [57]:
def P_X_CMND_vel_spheric(vx, vy, vz, F):
    mode = F.get('mode')
    center = F.get('center')
    
    if mode == 'volume_check':
        # f(x) = 1.0
        return np.ones_like(vx)
    
    elif mode == 'moment_check':
        # f(x) = distance_from_center^2
        # This tests if the Spherical Jacobian (rho^2 sin(theta)) is correct
        dist_sq = (vx - center[0])**2 + (vy - center[1])**2 + (vz - center[2])**2
        return dist_sq
    
    return np.zeros_like(vx)

In [58]:
def sphere_surface_integral_P_X_CMND(
    r: np.array,
    dr: float,
    c_v: np.array,
    r_v: float,
    deg: int,
    F: dict
) -> float:
    """
    Calculates the 6D integral of P_X_CMND using Gauss-Legendre Quadrature.
    Robust to P_X_CMND returning extra diagnostic values.
    """
    from numpy.polynomial.legendre import leggauss
    # 1. Get Gauss-Legendre nodes and weights
    nodes, weights = leggauss(deg)

    # 2. Velocity Mesh (Cartesian Box)
    x_vals = r[0] + dr * nodes
    y_vals = r[1] + dr * nodes
    z_vals = r[2] + dr * nodes
    
    # 3. Position Mesh (Spherical Coordinates)
    # rho: [0, r], theta: [0, pi], phi: [0, 2pi]
    rho_vals = (r_v / 2.0) * (nodes + 1)
    theta_vals = (np.pi / 2.0) * (nodes + 1)
    phi_vals = np.pi * (nodes + 1)

    # 4. Construct the 6D Meshgrid
    # Order: rho, theta, phi, vx, vy, vz
    Rho, Theta, Phi, X, Y, Z = np.meshgrid(
        rho_vals, theta_vals, phi_vals, x_vals, y_vals, z_vals, indexing='ij'
    )
    
    # Mesh the weights
    W_rho, W_theta, W_phi, W_vx, W_vy, W_vz = np.meshgrid(
        weights, weights, weights, weights, weights, weights, indexing='ij'
    )

    # 5. Coordinate Transformations
    sin_theta = np.sin(Theta)
    cos_theta = np.cos(Theta)
    sin_phi = np.sin(Phi)
    cos_phi = np.cos(Phi)
    
    VX = c_v[0] + Rho * sin_theta * cos_phi
    VY = c_v[1] + Rho * sin_theta * sin_phi
    VZ = c_v[2] + Rho * cos_theta

    # 6. Calculate Integration Weights (Jacobians)
    scale_factors = (r_v / 2.0) * (np.pi / 2.0) * np.pi * (dr**3)
    spherical_jacobian = (Rho**2) * sin_theta
    
    total_weights = (
        W_rho * W_theta * W_phi * W_vx * W_vy * W_vz 
        * spherical_jacobian 
        * scale_factors
    )

    # 7. Evaluate Function
    P_output = P_X_CMND_vel_spheric(
        VX.ravel(), VY.ravel(), VZ.ravel(), F
    )
    
    # --- FIX FOR TUPLE RETURN ---
    # If P_X_CMND returns (P, neas_count, etc.), take the first element.
    if isinstance(P_output, tuple):
        P_flat = P_output[0]
    else:
        P_flat = P_output
        
    # Ensure it is a numpy array
    P_flat = np.asarray(P_flat)
    # ----------------------------

    # 8. Integrate
    # Reshape P_flat to match the 6D grid shape of total_weights
    try:
        P_grid = P_flat.reshape(total_weights.shape)
    except ValueError as e:
        print(f"Shape Error Details: P_flat size {P_flat.size}, Weights shape {total_weights.shape}")
        raise e
    
    weighted_values = P_grid * total_weights
    
    integral_result = np.nansum(weighted_values)
    
    return integral_result

In [59]:
# Parameters
c_vel = np.array([10.0, -5.0, 2.0]) # Arbitrary center
r_vel = 2.5
c_pos = np.array([0.5, 0.5, 0.5])
dr_box = 1.2
degree = 10 # High precision

# --- TEST 1: Volume Check (f = 1) ---
# Analytical: (4/3 * pi * r^3) * (2*dv)^3
pos_sphere = (4/3) * np.pi * r_vel**3
vol_box = (2 * dr_box)**3
analytical_1 = pos_sphere * vol_box

# Run Numerical
result_1 = sphere_surface_integral_P_X_CMND(
    c_pos, 
    dr_box, 
    c_vel, 
    r_vel, 
    degree, 
    F={'mode': 'volume_check', 'center': c_vel}
) 

print("-" * 50)
print("TEST 1: Volume Check (f = 1)")
print(f"Analytical: {analytical_1:.10f}")
print(f"Numerical:  {result_1:.10f}")
print(f"Error:      {abs(analytical_1 - result_1):.10e}")

# --- TEST 2: Moment Check (f = rho^2) ---
# Analytical: (4/5 * pi * r^5) * (2*dv)^3
moment_sphere = (4/5) * np.pi * r_vel**5
analytical_2 = moment_sphere * vol_box

# Run Numerical
result_2 = sphere_surface_integral_P_X_CMND(
    c_pos, 
    dr_box, 
    c_vel, 
    r_vel, 
    degree, 
    F={'mode': 'moment_check', 'center': c_vel}
) 

print("-" * 50)
print("TEST 2: Moment Check (f = rho^2)")
print(f"Analytical: {analytical_2:.10f}")
print(f"Numerical:  {result_2:.10f}")
print(f"Error:      {abs(analytical_2 - result_2):.10e}")
print("-" * 50)

if abs(analytical_1 - result_1) < 1e-9 and abs(analytical_2 - result_2) < 1e-9:
    print("SUCCESS: The integrator is geometrically accurate.")
else:
    print("FAILURE: Check Jacobian or coordinate transformations.")

--------------------------------------------------
TEST 1: Volume Check (f = 1)
Analytical: 904.7786842339
Numerical:  904.7786842339
Error:      1.1368683772e-13
--------------------------------------------------
TEST 2: Moment Check (f = rho^2)
Analytical: 3392.9200658770
Numerical:  3392.9200658770
Error:      9.0949470177e-13
--------------------------------------------------
SUCCESS: The integrator is geometrically accurate.


### Testing spheric integration in positions and cube in velocities 

In [60]:
# --- 1. Define the Integrator (Your function, slightly cleaned up) ---
def calculate_surface_integral_P_X_CMND(c, r, v, dv, deg, q_max, e_max, i_max, mu, F):
    # Nodes and weights
    nodes, weights = leggauss(deg)

    # Velocity Mesh (Box)
    vx_vals = v[0] + dv * nodes
    vy_vals = v[1] + dv * nodes
    vz_vals = v[2] + dv * nodes
    
    # Position Mesh (Sphere)
    rho_vals = (r / 2.0) * (nodes + 1)
    theta_vals = (np.pi / 2.0) * (nodes + 1)
    phi_vals = np.pi * (nodes + 1)

    # 6D Meshgrid
    Rho, Theta, Phi, Vx, Vy, Vz = np.meshgrid(
        rho_vals, theta_vals, phi_vals, vx_vals, vy_vals, vz_vals, indexing='ij'
    )
    
    W_rho, W_theta, W_phi, W_vx, W_vy, W_vz = np.meshgrid(
        weights, weights, weights, weights, weights, weights, indexing='ij'
    )

    # Transform Spherical -> Cartesian
    sin_theta = np.sin(Theta)
    X = c[0] + Rho * sin_theta * np.cos(Phi)
    Y = c[1] + Rho * sin_theta * np.sin(Phi)
    Z = c[2] + Rho * np.cos(Theta)

    # Weights and Jacobian
    scale_factors = (r / 2.0) * (np.pi / 2.0) * np.pi * (dv**3)
    spherical_jacobian = (Rho**2) * sin_theta
    
    total_weights = (
        W_rho * W_theta * W_phi * W_vx * W_vy * W_vz 
        * spherical_jacobian 
        * scale_factors
    )

    # Call the Function (The Mock P_X_CMND defined below)
    # We pass 'mode' via the 'F' parameter to switch between test cases
    P_flat = P_X_CMND_pos_spheric(
        X.ravel(), Y.ravel(), Z.ravel(), 
        Vx.ravel(), Vy.ravel(), Vz.ravel(),
        q_max, e_max, i_max, mu, F
    )
    
    # Reshape and Integrate
    if isinstance(P_flat, tuple): P_flat = P_flat[0]
    P_grid = np.asarray(P_flat).reshape(total_weights.shape)
    
    return np.sum(P_grid * total_weights)

# --- 2. Define the Mock P_X_CMND Function ---
# This replaces your real function for the test. 
# We use the 'F' argument to tell it which math function to calculate.
def P_X_CMND_pos_spheric(x, y, z, vx, vy, vz, q_max, e_max, i_max, mu, F):
    mode = F.get('mode')
    center = F.get('center')
    
    if mode == 'volume_check':
        # f(x) = 1.0
        return np.ones_like(x)
    
    elif mode == 'moment_check':
        # f(x) = distance_from_center^2
        # This tests if the Spherical Jacobian (rho^2 sin(theta)) is correct
        dist_sq = (x - center[0])**2 + (y - center[1])**2 + (z - center[2])**2
        return dist_sq
    
    return np.zeros_like(x)

# --- 3. Run the Validation ---

# Parameters
c_pos = np.array([10.0, -5.0, 2.0]) # Arbitrary center
r_sphere = 2.5
c_vel = np.array([0.5, 0.5, 0.5])
dv_box = 1.2
degree = 10 # High precision

# --- TEST 1: Volume Check (f = 1) ---
# Analytical: (4/3 * pi * r^3) * (2*dv)^3
vol_sphere = (4/3) * np.pi * r_sphere**3
vol_box = (2 * dv_box)**3
analytical_1 = vol_sphere * vol_box

# Run Numerical
result_1 = calculate_surface_integral_P_X_CMND(
    c_pos, r_sphere, c_vel, dv_box, degree, 
    0, 0, 0, 0, 
    F={'mode': 'volume_check', 'center': c_pos}
)

print("-" * 50)
print("TEST 1: Volume Check (f = 1)")
print(f"Analytical: {analytical_1:.10f}")
print(f"Numerical:  {result_1:.10f}")
print(f"Error:      {abs(analytical_1 - result_1):.10e}")

# --- TEST 2: Moment Check (f = rho^2) ---
# Analytical: (4/5 * pi * r^5) * (2*dv)^3
moment_sphere = (4/5) * np.pi * r_sphere**5
analytical_2 = moment_sphere * vol_box

# Run Numerical
result_2 = calculate_surface_integral_P_X_CMND(
    c_pos, r_sphere, c_vel, dv_box, degree, 
    0, 0, 0, 0, 
    F={'mode': 'moment_check', 'center': c_pos}
)

print("-" * 50)
print("TEST 2: Moment Check (f = rho^2)")
print(f"Analytical: {analytical_2:.10f}")
print(f"Numerical:  {result_2:.10f}")
print(f"Error:      {abs(analytical_2 - result_2):.10e}")
print("-" * 50)

if abs(analytical_1 - result_1) < 1e-9 and abs(analytical_2 - result_2) < 1e-9:
    print("SUCCESS: The integrator is geometrically accurate.")
else:
    print("FAILURE: Check Jacobian or coordinate transformations.")

--------------------------------------------------
TEST 1: Volume Check (f = 1)
Analytical: 904.7786842339
Numerical:  904.7786842339
Error:      1.1368683772e-13
--------------------------------------------------
TEST 2: Moment Check (f = rho^2)
Analytical: 3392.9200658770
Numerical:  3392.9200658770
Error:      9.0949470177e-13
--------------------------------------------------
SUCCESS: The integrator is geometrically accurate.
